<div align='center'>

# 🧠 AquaVolt-AI — PIML Architecture Deep Dive
### Physics-Informed Neural Network Crop Coefficient Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umertanveer25/aquavolt-ai-pk/blob/main/notebooks/piml_architecture.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-aquavolt--ai--pk-black?logo=github)](https://github.com/umertanveer25/aquavolt-ai-pk)

**Umer Tanveer** · PhD Candidate, AWKUM Pakistan

</div>

---

This notebook provides a complete technical deep dive into the **Physics-Informed Machine Learning (PIML)** architecture used by AquaVolt-AI.

The system couples FAO-56 Penman-Monteith thermodynamics with a bounded neural residual corrector to estimate crop water requirements.

### Key Equations

**FAO-56 Reference Evapotranspiration:**

$$ET_0 = \frac{0.408\,\Delta\,(R_n - G) + \gamma\,\frac{900}{T+273}\,u_2\,(e_s - e_a)}{\Delta + \gamma\,(1 + 0.34\,u_2)}$$

**PIML Crop Coefficient:**

$$K_c = \text{clip}\!\left(K_{c,\text{prior}} + \text{clip}(r \cdot 0.15,\ -0.15,\ +0.15),\ 0.15,\ 1.20\right)$$

**Crop ET under Stress:**

$$ET_c = K_s \cdot K_c \cdot ET_0$$

In [ ]:
!pip install -q matplotlib seaborn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SHEET_ID = '1c2a-3t8fF2g_PX_0ape4ASTsbr5uX0Zb6YPzT8jtuN8'
url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=Sheet1'

print('🛰️  Loading live data...')
df = pd.read_csv(url, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
for c in ['ndvi','kc','ks','etc','water_need','air_temp','solar_rad','humidity']:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp','ndvi','kc'])

FIELD_COLORS = {
    'Field-A (Corn)': '#2196F3', 'Field-B (Alfalfa)': '#4CAF50',
    'Field-C (Fallow)': '#FF9800', 'Field-D (Tomato)': '#E91E63'}

print(f'✅ Loaded {len(df):,} records')

## 📐 Step 1: The Kc Sigmoid Prior Curve
The physics layer maps NDVI to a prior crop coefficient using a sigmoid function. This encodes the agronomic knowledge that bare soil (NDVI≈0) has low water demand, while dense vegetation (NDVI≈0.8+) has high demand.

In [ ]:
ndvi_range = np.linspace(0, 1, 500)
kc_prior = 0.15 + 0.95 / (1 + np.exp(-12 * (ndvi_range - 0.4)))
kc_upper = np.clip(kc_prior + 0.15, 0.15, 1.20)
kc_lower = np.clip(kc_prior - 0.15, 0.15, 1.20)

fig, ax = plt.subplots(figsize=(14, 7), facecolor='#0e1117')
ax.set_facecolor('#1a1a2e')

# Correction envelope
ax.fill_between(ndvi_range, kc_lower, kc_upper, alpha=0.15, color='#4fc3f7', label='±0.15 Neural Correction Envelope')

# Prior curve
ax.plot(ndvi_range, kc_prior, color='#4fc3f7', linewidth=3, label='Kc Physics Prior (Sigmoid)', zorder=5)

# FAO-56 bounds
ax.axhline(0.15, color='#ef5350', linestyle='--', linewidth=1.5, alpha=0.7, label='FAO-56 Min Kc = 0.15')
ax.axhline(1.20, color='#ef5350', linestyle='--', linewidth=1.5, alpha=0.7, label='FAO-56 Max Kc = 1.20')

# Scatter actual data
for field, color in FIELD_COLORS.items():
    fdata = df[df['field_name'] == field].sample(min(200, len(df[df['field_name']==field])))
    ax.scatter(fdata['ndvi'], fdata['kc'], color=color, alpha=0.5, s=15, label=field, zorder=4)

ax.set_xlabel('NDVI (Vegetation Index)', color='#90a4ae', fontsize=13)
ax.set_ylabel('Crop Coefficient (Kc)', color='#90a4ae', fontsize=13)
ax.set_title('🧠 Physics-Informed Kc Estimation: Sigmoid Prior + Neural Correction Envelope',
             color='white', fontsize=14, fontweight='bold', pad=15)
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10, loc='upper left')
ax.tick_params(colors='#90a4ae', labelsize=11)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.35)
ax.grid(True, alpha=0.1, color='#90a4ae')
for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')

plt.tight_layout()
plt.savefig('piml_sigmoid_prior.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

## 📊 Step 2: Neural Correction Distribution
The neural network predicts a correction $\delta K_c$ that is **hard-clipped** to ±0.15. This prevents the ML from ever overriding the physics.

In [ ]:
# Compute the physics prior for each data point
df['kc_prior'] = 0.15 + 0.95 / (1 + np.exp(-12 * (df['ndvi'] - 0.4)))
df['delta_kc'] = df['kc'] - df['kc_prior']

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0e1117')

# Histogram of corrections
ax = axes[0]
ax.set_facecolor('#1a1a2e')
for field, color in FIELD_COLORS.items():
    fdata = df[df['field_name'] == field]['delta_kc'].dropna()
    ax.hist(fdata, bins=30, color=color, alpha=0.5, label=field, edgecolor='white', linewidth=0.3)
ax.axvline(-0.15, color='#ef5350', linestyle='--', linewidth=2, label='Hard Clip Bound (±0.15)')
ax.axvline(0.15, color='#ef5350', linestyle='--', linewidth=2)
ax.axvline(0, color='white', linestyle='-', linewidth=1.5, alpha=0.5)
ax.set_xlabel('δKc (Neural Correction)', color='#90a4ae', fontsize=12)
ax.set_ylabel('Frequency', color='#90a4ae', fontsize=12)
ax.set_title('Neural Correction Distribution (Bounded ±0.15)', color='white', fontweight='bold')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax.tick_params(colors='#90a4ae')
for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')

# Ks distribution
ax = axes[1]
ax.set_facecolor('#1a1a2e')
for field, color in FIELD_COLORS.items():
    fdata = df[df['field_name'] == field]['ks'].dropna()
    ax.hist(fdata, bins=30, color=color, alpha=0.5, label=field, edgecolor='white', linewidth=0.3)
ax.axvline(1.0, color='#4fc3f7', linestyle='--', linewidth=1.5, label='No Stress (Ks=1.0)')
ax.set_xlabel('Ks (Water Stress Factor)', color='#90a4ae', fontsize=12)
ax.set_ylabel('Frequency', color='#90a4ae', fontsize=12)
ax.set_title('Water Stress Factor Distribution by Field', color='white', fontweight='bold')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax.tick_params(colors='#90a4ae')
for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')

plt.tight_layout()
plt.savefig('piml_corrections.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

## 🔬 Step 3: ETc Decomposition — Physics Component Analysis
Final crop ET is computed as: $ET_c = K_s \cdot K_c \cdot ET_0$. Let's visualize the contribution of each component.

In [ ]:
df['hour'] = df['timestamp'].dt.floor('h')
hourly = df.groupby(['hour','field_name']).agg(
    kc=('kc','mean'), ks=('ks','mean'), etc=('etc','mean'),
    ndvi=('ndvi','mean'), water_need=('water_need','mean')
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#0e1117')
axes = axes.flatten()

for i, (field, color) in enumerate(FIELD_COLORS.items()):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')
    fdata = hourly[hourly['field_name'] == field].sort_values('hour')
    
    ax.plot(fdata['hour'], fdata['kc'], color='#4fc3f7', linewidth=1.5, label='Kc (Crop Coeff)', alpha=0.9)
    ax.plot(fdata['hour'], fdata['ks'], color='#66bb6a', linewidth=1.5, label='Ks (Stress)', alpha=0.9)
    ax2 = ax.twinx()
    ax2.fill_between(fdata['hour'], fdata['etc'], color=color, alpha=0.2)
    ax2.plot(fdata['hour'], fdata['etc'], color=color, linewidth=2, label='ETc (mm/hr)')
    ax2.set_ylabel('ETc (mm/hr)', color=color, fontsize=10)
    ax2.tick_params(colors=color)
    
    ax.set_title(f'🌾 {field}', color='white', fontweight='bold', fontsize=11)
    ax.set_ylabel('Kc / Ks', color='#90a4ae')
    ax.tick_params(colors='#90a4ae', labelsize=8)
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 1.3)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, facecolor='#1a1a2e', labelcolor='white', fontsize=8, loc='upper right')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e3a5f')
    for sp in ax2.spines.values(): sp.set_edgecolor('#1e3a5f')

plt.suptitle('🔬 ETc Decomposition: Kc × Ks × ET₀ → ETc', color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('piml_etc_decomposition.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

## 🏗️ Step 4: PIML Pipeline Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4), facecolor='#0e1117')
ax.set_facecolor('#0e1117')
ax.axis('off')

boxes = [
    (0.05, 'Sentinel-2\nNDVI/NDWI', '#1565C0'),
    (0.22, 'Sigmoid\nKc Prior', '#00838F'),
    (0.39, 'Neural Net\nδKc (±0.15)', '#6A1B9A'),
    (0.56, 'Hard Clip\n[0.15, 1.20]', '#C62828'),
    (0.73, 'Ks × Kc × ET₀', '#2E7D32'),
    (0.90, 'ETc\n(mm/day)', '#E65100'),
]

for x, text, color in boxes:
    rect = mpatches.FancyBboxPatch((x, 0.25), 0.12, 0.5, boxstyle='round,pad=0.02',
                                   facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x + 0.06, 0.5, text, ha='center', va='center', color='white',
            fontsize=10, fontweight='bold')

for i in range(len(boxes) - 1):
    x1 = boxes[i][0] + 0.12
    x2 = boxes[i+1][0]
    ax.annotate('', xy=(x2, 0.5), xytext=(x1, 0.5),
                arrowprops=dict(arrowstyle='->', color='white', lw=2))

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1)
ax.set_title('🏗️ AquaVolt-AI PIML Pipeline', color='white', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('piml_pipeline.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

---

<div align='center'>

### 🧠 AquaVolt-AI — PIML Architecture Analysis Complete
*The neural corrector is strictly bounded — physics cannot be overridden*  
**Umer Tanveer** · PhD Candidate · AWKUM Pakistan  
[GitHub](https://github.com/umertanveer25/aquavolt-ai-pk)

</div>